In [0]:
# =============================================================================
# Notebook  : 02b_map_02_accounts_all
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02b_map_02_accounts_all
# Spec Ref  : §1.2 Refresh Materialized Views
# Purpose   : Build accounts_all Delta table = accounts UNION ALL crm_accounts
#             (currently just hgi.silver.accounts — ready for multi-source union)
#             Run AFTER map_01 (contacts_all must exist first).
#
# Serverless: works on 2X-Small
# Job Compute: full perf configs applied
# =============================================================================

In [0]:
# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/pipeline_config.py

In [0]:

%run ./_map_common.py

In [0]:
# CELL 2 ── Widget
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").strip().lower()

print(f"=== Map 02: accounts_all ===  customer={customer_id}")
print(f"  Reading from : {sv}.accounts")
print(f"  Writing to   : {sv}.accounts_all")

In [0]:
# CELL 3 ── Build accounts_all Delta table
# CLUSTER BY (tenant, domain) so Phase 2 of contacts_to_accounts (domain matching)
# can skip non-matching domain files — critical for performance on large accounts.

# Safe drop in case target exists as a VIEW
safe_drop_view(f"{sv}.accounts_all")

spark.sql(f"""
    CREATE OR REPLACE TABLE {sv}.accounts_all
    USING DELTA
    CLUSTER BY (tenant, domain)
    {DELTA_TBLPROPS_MAP}
    AS
    SELECT
        id,
        tenant,
        domain,
        name,
        source_system,
        source_system_object,
        source_key_name,
        source_key_value,
        data_timestamp
    FROM {sv}.accounts
    WHERE id IS NOT NULL
""")


In [0]:
# CELL 4 ── Verify
n = cnt(f"{sv}.accounts_all")
print(f"  accounts_all: {n:,} rows built")

# Spec DQ gate: domain must be lowercase
bad_domain = spark.sql(f"""
    SELECT COUNT(*) FROM {sv}.accounts_all
    WHERE domain IS NOT NULL AND domain != LOWER(domain)
""").collect()[0][0]
if bad_domain > 0:
    print(f"  WARNING: {bad_domain} accounts with uppercase domain")
else:
    print(f"  DQ OK: all domains lowercase")

dbutils.notebook.exit("Success")
